# 10 세 버전의 실험 비교와 배포 전 검증

**상황:** “금요일 오후, 새 상담 AI를 배포해도 될까?” 3개 정상 질문과 재고 실패 질문 1개로 변경을 검증합니다.

**25분 핵심:** 1–7단계. Docker·모델은 수업 전에 준비합니다. 기본 실행은 모델 호출 총 24회(실패 요청 2 + 기준선 8 + 프롬프트 개선 8 + 기능 정책 6), 동시성 1입니다.

A 기준 프롬프트 / B 근거 제한 프롬프트 / C 근거 제한 + 기능 정책.
C의 실시간 조회 요청은 실제 서비스 정책으로 중단하며 모델 호출을 수행하지 않습니다. 가짜 모델 응답이 아닙니다.
같은 4개 사례와 같은 모델을 사용하고, 모델의 오판을 포함한 원본 결과를 보존합니다.

**수업 전 준비:** [Docker·VS Code 안내](../langfuse/README.md)의 명령을 터미널에서 마친 뒤 실행합니다.

```sh
python -m pip install -r requirements/week11-langfuse.txt
python week11/lab/langfuse/init_env.py
docker compose -f week11/lab/langfuse/compose.yaml up -d --wait --wait-timeout 300
ollama pull qwen2.5:1.5b-instruct
```

Ollama 앱과 Docker Desktop이 실행 중이어야 합니다. [Langfuse 화면](http://localhost:3011)에 접속하고
`week11/lab/langfuse/.env`의 `LANGFUSE_LOGIN_EMAIL`, `LANGFUSE_LOGIN_PASSWORD`로 로그인합니다.
이 파일은 본인 PC의 비공개 설정입니다. 노트북 출력이나 GitHub에 키·비밀번호를 복사하지 않습니다.

`.venv` Python 3.11 커널에서 위부터 Shift+Enter. 다른 Notebook의 변수는 필요 없습니다.
모델의 의미 오류는 관찰 대상이며, 연결 실패·파싱 오류를 가짜 성공으로 대체하지 않습니다.


In [ ]:
from pathlib import Path
import sys, json
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'week11/lab/langfuse_lab.py').exists())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
assert sys.version_info[:2] == (3, 11), 'Python 3.11 .venv 커널을 선택하세요.'
import pandas as pd
from IPython.display import display, Markdown
from week11.lab.evaluation import load_data
from week11.lab.ollama_lab import OllamaClient
from week11.lab.langfuse_lab import connect_langfuse, traced_question, wait_for_trace
lf = connect_langfuse()  # 이 실습 폴더의 .env만 사용. 키를 출력하지 않음.
ollama = OllamaClient()
print(ollama.probe())
by_id = {q['id']: q for q in load_data()[1]}


## 1. 실패 관찰과 개선 가설

재고 질문의 실제 응답을 확인합니다. 단일 질문을 본 뒤 전체 성능이 좋아졌다고 판단하지 않습니다.

In [ ]:
from week11.lab.release_lab import ensure_dataset, run_version, release_decision, DEFAULT_POLICY
incident = traced_question(lf, ollama, by_id['Q24a'], prompt_version='baseline')
wait_for_trace(lf, incident)
print(incident['answer'])
print('가설 B: 근거 제한 프롬프트로 개선 / 가설 C: 미지원 기능을 서비스에서 차단')

**확인 결과 / 화면에서 볼 것**

실제 답변이 우연히 올바르더라도 경계 사례로 유지합니다. 실패 사례만 모으지 않고 정상 사례도 함께 넣습니다.

## 2. 정상 사례와 실패 사례의 고정 데이터

내용 해시가 같은 데이터만 재사용합니다. 다른 사람이 바꾼 평가 기준은 자동 덮어쓰지 않습니다.

In [ ]:
dataset = ensure_dataset(lf, incident)
display(pd.DataFrame([{'id':i.input['question_id'], 'question':i.input['question'],
                      'request_type':i.input['requested_capability'], 'expected':i.expected_output} for i in dataset.items]))
print('데이터:', dataset.name)
POLICY = dict(DEFAULT_POLICY)  # 실험 결과를 보기 전에 고정
print('배포 전 조건:', POLICY)

**확인 결과 / 화면에서 볼 것**

Datasets에 4개 사례가 보입니다. Q24a는 source trace와 연결됩니다. requested_capability는 상담 화면의 메뉴/라우트 입력입니다. 정답의 answerable 값으로 서비스 분기를 선택하지 않으며, 자유 문장의 의도 분류는 이 실습 범위 밖입니다.

## 3. A 기준선의 실제 실험

같은 검색기와 모델을 사용합니다. 기본 동시성 1로 로컬 모델 큐 적체를 피합니다.

In [ ]:
baseline = run_version(lf, ollama, dataset, 'baseline')
display(Markdown(f"[A 기준선 실험 열기]({baseline['experiment_url']})"))
display(pd.DataFrame([{'id':r['question_id'], 'answer':r['output']['answer'],
                      'judge':r['output']['judge']} for r in baseline['rows']]))

**확인 결과 / 화면에서 볼 것**

Experiments에 기준선 실행 4개가 생깁니다. generation_ms, structural_pass, false_accept와 실제 답변을 함께 읽습니다.

## 4. B 프롬프트 개선의 실제 실험

평가 데이터와 모델을 고정하고 프롬프트만 변경합니다. 성공을 가정하지 않습니다.

In [ ]:
candidate = run_version(lf, ollama, dataset, 'grounded')
display(Markdown(f"[B 프롬프트 개선 실험 열기]({candidate['experiment_url']})"))
decision_b = release_decision(baseline, candidate, POLICY)
print(decision_b['decision'])
display(pd.DataFrame(decision_b['checks']))

**확인 결과 / 화면에서 볼 것**

실제 실행에서 더 좋아질 수도, 비슷하거나 나빠질 수도 있습니다. 문장은 확인 불가인데 기권 필드가 틀린 사례를 찾아봅니다.

## 5. C 기능 정책의 실제 적용

policy_qa만 구현된 서비스에서는 live_inventory를 조회할 수 없습니다. 서비스의 능력 범위를 코드로 제한합니다.

In [ ]:
from week11.lab.release_lab import capability_response
print('정책 안내:', capability_response('policy_qa'))
print('미지원 실시간 조회:', capability_response('live_inventory'))
guarded = run_version(lf, ollama, dataset, 'capability_guard')
display(Markdown(f"[C 기능 정책 실험 열기]({guarded['experiment_url']})"))
decision_c = release_decision(baseline, guarded, POLICY)
print(decision_c['decision'])
display(pd.DataFrame(decision_c['checks']))

**확인 결과 / 화면에서 볼 것**

Q24a trace에는 capability-check와 contract-checks가 보이고 모델 generation은 없습니다. 정책 응답임을 명시합니다. 정상 질문 3개는 여전히 실제 Ollama 생성·평가를 실행합니다.

## 6. 한 화면에서 비교하고 배포 보류·검토 결정

평균 점수가 필수 실패를 상쇄하지 않도록 개별 조건을 확인합니다. 4건으로 p95나 일반 성능을 주장하지 않습니다.

In [ ]:
display(pd.DataFrame([
 {'version':'A 기준선', **decision_b['summary']['baseline']},
 {'version':'B 프롬프트', **decision_b['summary']['candidate']},
 {'version':'C 기능 정책', **decision_c['summary']['candidate']},
]))
for label,decision in [('B',decision_b),('C',decision_c)]:
    print(label,decision['decision'],decision['reason'])

**확인 결과 / 화면에서 볼 것**

HOLD는 배포 보류, REVIEW는 사람 검토 대상으로 이동입니다. REVIEW도 자동 배포가 아닙니다. 인용·기권 검사 통과율은 답변 의미 정확도와 다릅니다. 작은 모델의 판정은 필수 실패를 덮어쓰지 않습니다.

## 7. 서버에 저장된 실험과 점수 검증

반환된 Python 객체만 믿지 않고 실제 Langfuse API에서 각 요청을 재조회합니다.

In [ ]:
checks = []
for experiment in [baseline,candidate,guarded]:
    for row in experiment['rows']:
        saved = wait_for_trace(lf,row['output'])
        checks.append({'version':experiment['version'],'id':row['question_id'],
                       'observations':len(saved['observations']),'scores':len(saved['scores'])})
display(pd.DataFrame(checks))
print('3개 실험 × 4개 요청의 서버 저장 확인')

**확인 결과 / 화면에서 볼 것**

Experiments에서 같은 데이터의 A/B/C를 선택하여 비교하고, 실패 셀에서 개별 trace로 이동합니다. 서비스 기능 정책의 경로는 generation이 없는 것이 정상입니다.

## 8. 배포 판단 근거와 다음 실험 기록

관찰 → 실패 사례 → 고정 평가 → 변경 비교 → 배포 조건 → 사람 검토까지 연결합니다.

In [ ]:
out = ROOT/'week11/lab/reports/langfuse'
out.mkdir(parents=True,exist_ok=True)
report = {'baseline':baseline,'candidate':candidate,'guarded':guarded,'decision':decision_b,'guarded_decision':decision_c}
(out/'release-comparison.json').write_text(json.dumps(report,ensure_ascii=False,indent=2),encoding='utf-8')
print(out/'release-comparison.json')
print('다음 과제: 정상 답변의 조건 누락을 사람이 검토하고, 검토 근거와 다음 변경 하나를 제안')
lf.flush()

**확인 결과 / 화면에서 볼 것**

09번도 기록 저장을 마쳤다면 터미널에서 `python -m week11.lab.demo_guide`로 이번 실행의 화면별 바로가기를 만듭니다. `reports/langfuse/instructor-console.html`을 브라우저로 여세요. 새 마감 과제가 아닌 수업 활동입니다. CLI의 --enforce-gate는 B가 HOLD이면 종료 코드 2로 자동 차단합니다. 실제 운영 배포·롤백은 실행하지 않습니다. 3초 예산은 교육용 기본값이며 기기·서비스 조건에 맞게 실험 전에 정해야 합니다.

## 마무리

연결이 끝나면 `lf.flush()`로 전송합니다. 실제 저장은 API 재조회로 확인합니다. 모델·프롬프트·기기 상태에 따라 답변과 지연은 달라집니다.
수업 결과는 합성 사례이며 일반적인 서비스 정확도 또는 운영 배포 승인이 아닙니다.

Docker 중지(데이터 유지): `docker compose -f week11/lab/langfuse/compose.yaml stop`
재개: 같은 경로의 `up -d --wait`. **`down -v`는 데이터를 삭제하므로 일반 종료에 사용하지 않습니다.**
[화면별 시연 안내](https://smilesjcha.github.io/llmops-16week/week11/lab/demo/) / [100분 진행·문제 해결](../langfuse/README.md) / [전체 실행 순서](../WALKTHROUGH.md)
